# Cowboy Outfits Detection: 🤠 👢 🕶️ 🧥 🥋
   
   In this kernel, we will introduce: 

* how to generate the specific required folder structure for yolov5
* Transfer COCO to YOLOv5 annotations and reverse it
* Do your project with Weight and Biases
* Inference section
* tuning hyperparameter (TBD)
* model selection (TBD)
* bounding box augmentation (TBD)
* cross-validation (TBD)

In [1]:
%cd /kaggle/
%ls

/kaggle
input/  lib/  working/


In [2]:
!mkdir training
%cd training

/kaggle/training


In [3]:
# Download YOLOv5
!git clone https://github.com/ultralytics/yolov5  # clone repo
%cd yolov5
# Install dependencies
%pip install -qr requirements.txt  # install dependencies

%cd ../

%pip install --upgrade torch 
%pip install --upgrade torchvision

import torch
import torchvision

print(f"Setup complete. Using torch {torch.__version__}, torchvision {torchvision.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

Cloning into 'yolov5'...
remote: Enumerating objects: 8670, done.
remote: Counting objects: 100% (384/384), done.
remote: Compressing objects: 100% (250/250), done.
remote: Total 8670 (delta 234), reused 250 (delta 134), pack-reused 8286
Receiving objects: 100% (8670/8670), 9.67 MiB | 16.50 MiB/s, done.
Resolving deltas: 100% (5974/5974), done.
/kaggle/training/yolov5
Note: you may need to restart the kernel to use updated packages.
/kaggle/training
     |████████████████████████████████| 831.4 MB 1.1 kB/s  eta 0:00:01     |███████████████████████████████ | 806.1 MB 70.9 MB/s eta 0:00:01     |███████████████████████████████▋| 820.0 MB 70.9 MB/s eta 0:00:01     |███████████████████████████████▋| 821.2 MB 70.9 MB/s eta 0:00:01     |███████████████████████████████▋| 822.5 MB 70.9 MB/s eta 0:00:01
  Attempting uninstall: torch
    Found existing installation: torch 1.7.0
    Uninstalling torch-1.7.0:
      Successfully uninstalled torch-1.7.0
ERROR: pip's dependency resolver does not curre

# Manage your experiment with W&B

Weights and Biases (W&B) is a power tool which can help us do the machine learning experiment tracking, dataset versioning, and model evaluation. If you had already worked with tensorboard before, then the W&B should be easy for you.
[check the official documentation for more information](https://docs.wandb.ai/). [中文文档](https://docs.wandb.ai/v/zh-hans/)

The W&B had arealdy integrated into the latest `yolov5` We can easily train our model with it.

Since it is an online tool, if you can not get access to interent or you have some privacy concerns, yolov5 can also works with tensorboard. 

In [ ]:
# # Start tensorboard
# # Launch after you have started training to all the graphs needed for inspection

# since there are some probelem when using tensorbaord in kaggle, you can have a try with yourself with local machine.
# There is another ways to lunach the tensorboard feature, 
# just uncomment the tensorboard code after line 289 in the file `\yolov5\models\yolo.py`, just check the code for detail

In [ ]:
# # logs save in the folder "yolov5/runs"  (option)
%load_ext tensorboard
%tensorboard --logdir /kaggle/training/yolov5/runs

If you don't have a wb account, you should create a new account

In [4]:
# Install W&B 
%pip install -q --upgrade wandb
# Login with token, follow with the guide
import wandb
wandb.login()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
allennlp 2.5.0 requires torch<1.9.0,>=1.6.0, but you have torch 1.9.0 which is incompatible.
allennlp 2.5.0 requires torchvision<0.10.0,>=0.8.1, but you have torchvision 0.10.0 which is incompatible.
allennlp 2.5.0 requires wandb<0.11.0,>=0.10.0, but you have wandb 0.11.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


wandb: You can find your API key in your browser here: https://wandb.ai/authorize


wandb: Paste an API key from your profile and hit enter:  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [5]:
# Import lib
import os
import gc
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import yaml
from shutil import copyfile
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [6]:
%cd /kaggle/training/
%ls 

/kaggle/training
yolov5/


# Transfer annotations bbox from COCO to YOLO format

In [7]:
# Firstly, We will transfer the coco format to yolo format

#load json file
json_file_path = '/kaggle/input/cowboyoutfits/train.json'

data = json.load(open(json_file_path, 'r'))
yolo_anno_path = '/kaggle/training/yolo_anno/'

if not os.path.exists(yolo_anno_path):
    os.makedirs(yolo_anno_path)

there are tons of information in the json file,  you should check it by your self. 

>Example code like: 
`data['info']`
`data['image']`
`data['annotations']`
`data['categories']`

It can help us get better understand about the json structure. Some features might help like `data['annotations'][i][iscrowd]`. 
If possible we can do another EDA kernel for this match :D

In [8]:
# 需要注意下因为我们的annotation lable是不连续的,会导致后面报错,所以这里生成一个map映射
cate_id_map = {}
num = 0
for cate in data['categories']:
    cate_id_map[cate['id']] = num
    num+=1

In [9]:
cate_id_map

{87: 0, 1034: 1, 131: 2, 318: 3, 588: 4}

In [10]:
#对比下
data['categories']

[{'id': 87, 'name': 'belt', 'freebase_id': '/m/0176mf'},
 {'id': 1034, 'name': 'sunglasses', 'freebase_id': '/m/017ftj'},
 {'id': 131, 'name': 'boot', 'freebase_id': '/m/01b638'},
 {'id': 318, 'name': 'cowboy_hat', 'freebase_id': '/m/025rp__'},
 {'id': 588, 'name': 'jacket', 'freebase_id': '/m/032b3c'}]

In [11]:
# convert the bounding box from COCO to YOLO format.

def cc2yolo_bbox(img_width, img_height, bbox):
    dw = 1. / img_width
    dh = 1. / img_height
    x = bbox[0] + bbox[2] / 2.0
    y = bbox[1] + bbox[3] / 2.0
    w = bbox[2]
    h = bbox[3]
 
    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh
    return (x, y, w, h)

In [12]:
# transfer the annotation, and generated a train dataframe file
f = open('train.csv','w')
f.write('id,file_name\n')
for i in tqdm(range(len(data['images']))):
    filename = data['images'][i]['file_name']
    img_width = data['images'][i]['width']
    img_height = data['images'][i]['height']
    img_id = data['images'][i]['id']
    yolo_txt_name = filename.split('.')[0] + '.txt' #remove .jpg
    
    f.write('{},{}\n'.format(img_id, filename))
    yolo_txt_file = open(os.path.join(yolo_anno_path, yolo_txt_name), 'w')
    
    for anno in data['annotations']:
        if anno['image_id'] == img_id:
            yolo_bbox = cc2yolo_bbox(img_width, img_height, anno['bbox']) # "bbox": [x,y,width,height]        
            yolo_txt_file.write('{} {} {} {} {}\n'.format(cate_id_map[anno['category_id']], yolo_bbox[0], yolo_bbox[1], yolo_bbox[2], yolo_bbox[3]))
    yolo_txt_file.close()
f.close()

100%|██████████| 3062/3062 [00:03<00:00, 951.22it/s] 


In [13]:
# generate training dataframe
train = pd.read_csv('/kaggle/training/train.csv')
train.head()

,id,file_name
0,9860841628484337660,88d8bf3754317ffc.jpg
1,15984033263460081658,ddd2b190ea90dffa.jpg
2,76077631043502082,010e4833cdb38002.jpg
3,18065680256228130812,fab6307a1a43fffc.jpg
4,9491379842992996352,83b827ae01e68000.jpg


# Splitting data into training and validation

In [14]:
train_df, valid_df = train_test_split(train, test_size=0.10, random_state=233)

print(f'Size of total training images: {len(train)}, training images: {len(train_df)}. validation images: {len(valid_df)}')

# 说明下，这里给的validation set 就0.01，是因为这次的比赛挑战之一就是数据集很小，还是希望能够给更多的数据来训练。
# 其次，老师已经划分好了validation集了，这里的0.01的验证更多是测试自己算法是否有错误，真正看performance还是提交上去看好。
# 100次的提交也足够验证performance了.

Size of total training images: 3062, training images: 2755. validation images: 307


In [15]:
# generate new train data frame with spliter mark

train_df.loc[:, 'split'] = 'train'
valid_df.loc[:, 'split'] = 'valid'
df = pd.concat([train_df, valid_df]).reset_index(drop=True)
df.sample(10)

/opt/conda/lib/python3.7/site-packages/pandas/core/indexing.py:1597: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = value
/opt/conda/lib/python3.7/site-packages/pandas/core/indexing.py:1720: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


,id,file_name,split
1506,2619649989875010464,245addc94a5b0ba0.jpg,train
533,10326149593370067655,8f4dda584a5e02c7.jpg,train
2912,14292777138545234433,c65a2514c4d01201.jpg,valid
672,18270278461970481659,fd8d1176093881fb.jpg,train
2090,9483462048911133233,839c067cea3f0a31.jpg,train
2299,1392571838109188872,135368b566b60308.jpg,train
2911,7276767453136389232,64fc4571d5518470.jpg,valid
2612,14011911676768487246,c2744f6e85ef074e.jpg,train
1001,6350019850813212462,581fcd691466872e.jpg,train
960,8335597257553707237,73adfd8ab73580e5.jpg,train


# Prepare required data folder structure 

The Yolov5 requires a specific directory structure for custom training, you can get more information from the [official example](https://github.com/ultralytics/yolov5/wiki/Train-Custom-Data)

```
/training    --temp_traning space
    /dataset --dataset fold, split into training and validation
         /images
         /labels
    /yolov5  --yolov5 main
```

In [16]:
%cd /kaggle/training/
%ls

/kaggle/training
train.csv  yolo_anno/  yolov5/


In [17]:
# mdke directory for traning section
os.makedirs('/kaggle/training/cowboy/images/train', exist_ok=True)
os.makedirs('/kaggle/training/cowboy/images/valid', exist_ok=True)

os.makedirs('/kaggle/training/cowboy/labels/train', exist_ok=True)
os.makedirs('/kaggle/training/cowboy/labels/valid', exist_ok=True)

%ls

cowboy/  train.csv  yolo_anno/  yolov5/


In [18]:
# move the images and annotations to relevant splited folders

for i in tqdm(range(len(df))):
    row = df.loc[i]
    name = row.file_name.split('.')[0]
    if row.split == 'train':
        copyfile(f'/kaggle/input/cowboyoutfits/images/{name}.jpg', f'/kaggle/training/cowboy/images/train/{name}.jpg')
        copyfile(f'/kaggle/training/yolo_anno/{name}.txt', f'/kaggle/training/cowboy/labels/train/{name}.txt')
    else:
        copyfile(f'/kaggle/input/cowboyoutfits/images/{name}.jpg', f'/kaggle/training/cowboy/images/valid/{name}.jpg')
        copyfile(f'/kaggle/training/yolo_anno/{name}.txt', f'/kaggle/training/cowboy/labels/valid/{name}.txt')

100%|██████████| 3062/3062 [00:33<00:00, 90.07it/s] 


# Create .yaml file

The `data.yaml` file is the dataset configuration file that contains information about the datast like path of images, annotaions. 
We should specific the following items:
*     the path of our training and validation data
*     the number of classes to be detected
*     the names corresponding to those classes

> ‼️ in this competition, the categories_id provided in the train.json is not a sequential number, we need re_map it.or we will get an error here.

> !! we can put our `YAML` file anywhere, since we can reference the path later, but can also simply put it in the `data` folder under `yolov5`

In [19]:
# Create  yaml file

data_yaml = dict(
    train = '../cowboy/images/train/',
    val = '../cowboy/images/valid',
    nc = 5,
    names = ['belt', 'sunglasses', 'boot', 'cowboy_hat', 'jacket']
)

# we will make the file under the yolov5/data/ directory.
with open('/kaggle/training/yolov5/data/data.yaml', 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=True)
    
%cat /kaggle/training/yolov5/data/data.yaml # show your YAML file

{names: [belt, sunglasses, boot, cowboy_hat, jacket], nc: 5, train: ../cowboy/images/train/,
  val: ../cowboy/images/valid}


# Training Hyperparameters

There are about 25 hyperparameters in tranining setting.You can get access it from `yolov5/data/hyps/hyp.scratch.yaml`. Here is useful [discussion](https://github.com/ultralytics/yolov5/issues/607) about the hyperparameters in yolov5. 
Here is the example content:

In [ ]:
# Hyperparameters for COCO training from scratch
# python train.py --batch 40 --cfg yolov5m.yaml --weights '' --data coco.yaml --img 640 --epochs 300
# See tutorials for hyperparameter evolution https://github.com/ultralytics/yolov5#tutorials


lr0: 0.01  # initial learning rate (SGD=1E-2, Adam=1E-3)
lrf: 0.2  # final OneCycleLR learning rate (lr0 * lrf)
momentum: 0.937  # SGD momentum/Adam beta1
weight_decay: 0.0005  # optimizer weight decay 5e-4
warmup_epochs: 3.0  # warmup epochs (fractions ok)
warmup_momentum: 0.8  # warmup initial momentum
warmup_bias_lr: 0.1  # warmup initial bias lr
box: 0.05  # box loss gain
cls: 0.5  # cls loss gain
cls_pw: 1.0  # cls BCELoss positive_weight
obj: 1.0  # obj loss gain (scale with pixels)
obj_pw: 1.0  # obj BCELoss positive_weight
iou_t: 0.20  # IoU training threshold
anchor_t: 4.0  # anchor-multiple threshold
# anchors: 3  # anchors per output layer (0 to ignore)
fl_gamma: 0.0  # focal loss gamma (efficientDet default gamma=1.5)
hsv_h: 0.015  # image HSV-Hue augmentation (fraction)
hsv_s: 0.7  # image HSV-Saturation augmentation (fraction)
hsv_v: 0.4  # image HSV-Value augmentation (fraction)
degrees: 0.0  # image rotation (+/- deg)
translate: 0.1  # image translation (+/- fraction)
scale: 0.5  # image scale (+/- gain)
shear: 0.0  # image shear (+/- deg)
perspective: 0.0  # image perspective (+/- fraction), range 0-0.001
flipud: 0.0  # image flip up-down (probability)
fliplr: 0.5  # image flip left-right (probability)
mosaic: 1.0  # image mosaic (probability)
mixup: 0.0  # image mixup (probability)
copy_paste: 0.0  # segment copy-paste (probability)


If you are new to yolo like me, we can use the default values, and choose some higher level parameters for our traning.

* --img {IMG_SIZE} \ # Input image size.
* --batch {BATCH_SIZE} \ # Batch size
* --epochs {EPOCHS} \ # Number of epochs
* --data data.yaml \ # Configuration file
* --weights yolov5s.pt \ # Model name
* --save_period 1\ # Save model after interval
* --project kaggle-cow-boy # W&B project name
* --name exp # experiment name

Yolo provided a lot of differen [pretrained models](https://github.com/ultralytics/yolov5#pretrained-checkpoints). Here we will use  the smallest one for study purpose.

* YOLOv5s
* YOLOv5m
* YOLOv5l
* YOLOv5x
etc…

In [26]:
#IMG_SIZE = 640  # the default image size in yolo is 640, it will automated resize our image during training and valudation.
BATCH_SIZE = 32 # wisely choose, use the largest size that can feed up all your gpu ram
EPOCHS = 5
MODEL = 'yolov5m.pt'  # 5s, 5m 5l
name = f'{MODEL}_BS_{BATCH_SIZE}_EP_{EPOCHS}'

# GO TRAIN

In [20]:
# we are ready to training our model with w&b
%cd /kaggle/training/yolov5/

/kaggle/training/yolov5


During training you can open your [project webpage](https://wandb.ai/momo233/kaggle-cwoboy) to visualize your training process.

#H Hyperparameter evolution


In [28]:
!python train.py --batch {BATCH_SIZE} \
                 --epochs {EPOCHS} \
                 --data data.yaml \
                 --weights {MODEL} \
                 --save_period 1 \
                 --project /kaggle/working/kaggle-cwoboy \
                 --name {name} \
                 --cache-images

train: weights=yolov5m.pt, cfg=, data=data.yaml, hyp=data/hyps/hyp.scratch.yaml, epochs=5, batch_size=32, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, evolve=None, bucket=, cache_images=True, image_weights=False, device=, multi_scale=False, single_cls=False, adam=False, sync_bn=False, workers=8, project=/kaggle/working/kaggle-cwoboy, entity=None, name=yolov5m.pt_BS_32_EP_5, exist_ok=False, quad=False, linear_lr=False, label_smoothing=0.0, upload_dataset=False, bbox_interval=-1, save_period=1, artifact_alias=latest, local_rank=-1, freeze=0
github: up to date with https://github.com/ultralytics/yolov5 ✅
2021-08-02 03:59:46.073557: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
wandb: Currently logged in as: momo233 (use `wandb login --relogin` to force relogin)
2021-08-02 03:59:51.116582: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic 

In [29]:
# Training result will be showed online with W&B

# Here we can also zip our training result for local visualization

!zip -r /kaggle/working/output.zip /kaggle/working/kaggle-cwoboy


  adding: kaggle/working/kaggle-cwoboy/ (stored 0%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/ (stored 0%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/val_batch0_labels.jpg (deflated 5%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/F1_curve.png (deflated 9%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/train_batch0.jpg (deflated 2%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/val_batch1_labels.jpg (deflated 4%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/labels_correlogram.jpg (deflated 24%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/opt.yaml (deflated 46%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/confusion_matrix.png (deflated 27%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/results.png (deflated 10%)
  adding: kaggle/working/kaggle-cwoboy/yolov5m.pt_BS_32_EP_5/events.out.tfevents.1627876789.d268f46e2e24.515.0 (deflated 30%)
  add

Result are all in the [project page in W&B here](https://wandb.ai/momo233/kaggle-cwoboy). Here is an eaxmple screenshot from my result, there are a lot of usefull information in W&B

<img src="https://api.wandb.ai/files/momo233/kaggle-cwoboy/2rgyap5g/media/images/Results_20_0.png" alt="drawing" width="800"/>


# Inference Section

Once we finished the training section, we should refer to [W&B Artifacts tab](https://wandb.ai/momo233/kaggle-cwoboy/artifacts/model/run_2rgyap5g_model/4141ab8a754564a37bb1) to choose our target model for inference.

>Download the `best` performance model and upload to kaggle.

In [ ]:
# In the Dev phase, we will only use the valid data for predicition. 
# Don't forget change it to test data in the Final pahse.

valid_df = pd.read_csv('/kaggle/input/cowboyoutfits/valid.csv')
test_df = pd.read_csv('/kaggle/input/cowboyoutfits/test.csv')
valid_df.head()

In [ ]:
valid_df.shape

In [ ]:
%cd /kaggle/training/
%ls

In [ ]:
# make directory to store the validation data.
os.makedirs('/kaggle/inference/valid', exist_ok=True)
os.makedirs('/kaggle/inference/test', exist_ok=True)

In [ ]:
# copy the validation image to inference folder for detection process
for i in tqdm(range(len(valid_df))):
    row = valid_df.loc[i]
    name = row.file_name.split('.')[0]
    copyfile(f'/kaggle/input/cowboyoutfits/images/{name}.jpg', f'/kaggle/inference/valid/{name}.jpg')

In [ ]:
VALID_PATH = '/kaggle/inference/valid/'
MODEL_PATH = '/kaggle/input/cowboy-object-detection-models/v0_ep20_best.pt'
IMAGE_PATH = '/kaggle/input/cowboyoutfits/images/'

# Inference Hyperparameter
YOLOv5 also provided a lot of hyperparameters for inference process, we can check the detect.py file for detail

Here I list some parameters:
* --weights {MODEL_PATH} \ # path to the best model.
* --source {TEST_PATH} \ # absolute path to the test images.
* --img {IMG_SIZE} \ # Size of image
* --conf 0.25 \ # Confidence threshold (default is 0.25)
* --iou-thres 0.45 \ # IOU threshold (default is 0.45)
* --max-det 100 \ # Number of detections per image (default is 1000) 
* --save-txt \ # Save predicted bounding box coordinates as txt files
* --save-conf # Save the confidence of prediction for each bounding box
* --augment # augmented inference, TTA
* --project 'runs/detect'  # save results to project/name
* --name 'exp'  # save results to project/name
* --half False  # use FP16 half-precision inference

We will use the default settings for inference. Here, we will simply introduced one example to pick the right confidence score which is based on our F1 score which is showed below. It can be access from our W&B result page. You can get access it from [here](https://wandb.ai/momo233/kaggle-cwoboy/runs/2rgyap5g?workspace=user-momo233). From the F1 score, we just pick the confidence with 0.546. it seems a little bit high, but can cover all the classes.

<img src="https://api.wandb.ai/files/momo233/kaggle-cwoboy/2rgyap5g/media/images/Results_20_2.png" alt="drawing" width="600"/>

# GO Detection

In [ ]:
# go to yolov5 main folder for detection
%cd /kaggle/training/yolov5/

In [ ]:
!python detect.py --weights {MODEL_PATH} \
                  --source {VALID_PATH} \
                  --conf 0.546 \
                  --iou-thres 0.5 \
                  --save-txt \
                  --save-conf \
                  --augment

In [ ]:
# read the output log , indicated our prediction result was saved under `runs/detect/exp/`

PRED_PATH = '/kaggle/training/yolov5/runs/detect/exp/labels/'

## visualize our prediction

In [ ]:
with open('/kaggle/training/yolov5/runs/detect/exp/labels/010fb53ff39a0ea1.txt', 'r') as file:
    for line in file:
        print(line)

In [ ]:
from PIL import Image
Image.open('/kaggle/training/yolov5/runs/detect/exp/010fb53ff39a0ea1.jpg')

# Make Submission

In [ ]:
# list our prediction files path
prediction_files = os.listdir(PRED_PATH)
print('Number of test images with detections: ', len(prediction_files))

In [ ]:
# convert yolo to coco annotation format
def yolo2cc_bbox(img_width, img_height, bbox):
    x = (bbox[0] - bbox[2] * 0.5) * img_width
    y = (bbox[1] - bbox[3] * 0.5) * img_height
    w = bbox[2] * img_width
    h = bbox[3] * img_height
    
    return (x, y, w, h)

In [ ]:
# reverse the categories numer to the origin id
re_cate_id_map = dict(zip(cate_id_map.values(), cate_id_map.keys()))

print(re_cate_id_map)

In [ ]:
def make_submission(df, PRED_PATH, IMAGE_PATH):
    output = []
    for i in tqdm(range(len(df))):
        row = df.loc[i]
        image_id = row['id']
        file_name = row['file_name'].split('.')[0]
        if f'{file_name}.txt' in prediction_files:
            img = Image.open(f'{IMAGE_PATH}/{file_name}.jpg')
            width, height = img.size
            with open(f'{PRED_PATH}/{file_name}.txt', 'r') as file:
                for line in file:
                    preds = line.strip('\n').split(' ')
                    preds = list(map(float, preds)) #conver string to float
                    cc_bbox = yolo2cc_bbox(width, height, preds[1:-1])
                    result = {
                        'image_id': image_id,
                        'category_id': re_cate_id_map[preds[0]],
                        'bbox': cc_bbox,
                        'score': preds[-1]
                    }

                    output.append(result)
    return output

In [ ]:
sub_data = make_submission(valid_df, PRED_PATH, IMAGE_PATH)

In [ ]:
op_pd = pd.DataFrame(sub_data)

op_pd.sample(10)

In [ ]:
import zipfile 

op_pd.to_json('/kaggle/working/answer.json',orient='records')
zf = zipfile.ZipFile('/kaggle/working/sample_answer.zip', 'w')
zf.write('/kaggle/working/answer.json', 'answer.json')
zf.close()

具体运行的结果可以去[W&B的项目页面](https://wandb.ai/momo233/kaggle-cwoboy)查看。因为kaggle save老是出错，所以这次就没有train, 只做了inference. 可以看version 3, 有第一次的train 的log.
虽然后面测试了不同模型，但是这里我都是使用了第一次train的model，然后更改了不同的conf 和 iou threat 参数，来查看相同模型的performace。 以后再更新不同模型好了。

| Model | train_setting | inference_setting| result|
| :-----| :---- | :---- | :---- |
| Yolov5s | epoch[20] bs[16] | conf[0.546]   iou[0.5] TTA[false]| [50.1540154015] |
| Yolov5s | epoch[20] bs[16] | conf[0.546]   iou[0.5] TTA[true]| [57.9936350778] |
| Yolov5s | epoch[20] bs[16] | conf[0.5]     iou[0.5] TTA[true]| [58.7389096052] |


因为也是第一次用yolo，还有很多地方不懂。目前几乎都是初始设置跑通一下项目而已。 

欢迎评论留言，和提出建议😃